In [0]:
# Databricks notebook source
# COMMAND ----------
# MAGIC %run ../00_setup/00_adls_gen2_oauth_connection

# COMMAND ----------
from datetime import datetime
from pyspark.sql.functions import current_timestamp, input_file_name, lit
import os
import shutil
import tempfile
import zipfile

# COMMAND ----------
# Parâmetros
storage_account = "stspmobilitydev001"
container = "bronze"

ingestion_date = datetime.now().strftime("%Y-%m-%d")
run_id = datetime.now().strftime("%Y%m%d%H%M%S")

raw_base_path = (
    f"abfss://{container}@{storage_account}.dfs.core.windows.net/gtfs"
)

zip_source_path = (
    f"{raw_base_path}/raw/cittamobi_gtfs.zip"
)

extract_base_local = f"/tmp/gtfs_extract_{run_id}"
zip_local_path = f"/tmp/cittamobi_gtfs_{run_id}.zip"

target_paths = {
    "routes": f"{raw_base_path}/routes",
    "trips": f"{raw_base_path}/trips",
    "stops": f"{raw_base_path}/stops",
    "stop_times": f"{raw_base_path}/stop_times",
    "calendar": f"{raw_base_path}/calendar",
    "shapes": f"{raw_base_path}/shapes",
}

required_files = {
    "routes": "routes.txt",
    "trips": "trips.txt",
    "stops": "stops.txt",
    "stop_times": "stop_times.txt",
    "calendar": "calendar.txt",
    "shapes": "shapes.txt",
}

print("=== GTFS STATIC INGESTION START ===")
print(f"storage_account: {storage_account}")
print(f"container: {container}")
print(f"ingestion_date: {ingestion_date}")
print(f"zip_source_path: {zip_source_path}")

# COMMAND ----------
def path_exists(path: str) -> bool:
    try:
        dbutils.fs.ls(path)
        return True
    except Exception:
        return False


def ensure_local_dir(path: str) -> None:
    if os.path.exists(path):
        shutil.rmtree(path, ignore_errors=True)
    os.makedirs(path, exist_ok=True)


def cleanup_local_paths(*paths: str) -> None:
    for path in paths:
        try:
            if os.path.isdir(path):
                shutil.rmtree(path, ignore_errors=True)
            elif os.path.isfile(path):
                os.remove(path)
        except Exception as e:
            print(f"[WARN] Falha ao limpar caminho local {path}: {e}")


# COMMAND ----------
# Validação de acesso ao storage
print("Validando acesso ao ADLS...")
root_ls = dbutils.fs.ls(f"abfss://{container}@{storage_account}.dfs.core.windows.net/")
print(f"Acesso OK. Itens encontrados na raiz do container {container}: {len(root_ls)}")

if not path_exists(zip_source_path):
    raise FileNotFoundError(f"ZIP GTFS não encontrado em: {zip_source_path}")

print("Arquivo ZIP encontrado no storage.")

# COMMAND ----------
# Preparação local
print("Preparando diretórios locais...")
cleanup_local_paths(zip_local_path, extract_base_local)
ensure_local_dir(extract_base_local)

# Copia ZIP do ADLS para o disco local do driver
print("Copiando ZIP do ADLS para o driver local...")
dbutils.fs.cp(zip_source_path, f"file:{zip_local_path}")
print(f"ZIP copiado para: {zip_local_path}")

# COMMAND ----------
# Extração local
print("Extraindo ZIP localmente...")
with zipfile.ZipFile(zip_local_path, "r") as zip_ref:
    zip_ref.extractall(extract_base_local)

extracted_files = sorted(os.listdir(extract_base_local))
print(f"Arquivos extraídos: {extracted_files}")

# Validação dos arquivos obrigatórios
missing_files = [
    filename for filename in required_files.values()
    if not os.path.exists(os.path.join(extract_base_local, filename))
]

if missing_files:
    raise FileNotFoundError(
        f"Arquivos obrigatórios ausentes no ZIP: {missing_files}"
    )

print("Todos os arquivos obrigatórios do GTFS foram encontrados.")

# COMMAND ----------
# Upload dos arquivos TXT extraídos para o ADLS
print("Enviando arquivos extraídos para o ADLS...")

for entity, filename in required_files.items():
    local_file = os.path.join(extract_base_local, filename)
    target_dir = target_paths[entity]
    target_file = f"{target_dir}/{filename}"

    print(f"Enviando {filename} -> {target_file}")
    dbutils.fs.cp(f"file:{local_file}", target_file, True)

print("Upload dos arquivos extraídos concluído.")

# COMMAND ----------
# Leitura dos arquivos GTFS a partir do ADLS
print("Lendo arquivos GTFS do ADLS...")

read_options = {
    "header": "true",
    "inferSchema": "true",
    "sep": ",",
}

routes_df = (
    spark.read.options(**read_options)
    .csv(f"{target_paths['routes']}/routes.txt")
    .withColumn("ingestion_date", lit(ingestion_date))
    .withColumn("ingested_at", current_timestamp())
    .withColumn("source_file", input_file_name())
)

trips_df = (
    spark.read.options(**read_options)
    .csv(f"{target_paths['trips']}/trips.txt")
    .withColumn("ingestion_date", lit(ingestion_date))
    .withColumn("ingested_at", current_timestamp())
    .withColumn("source_file", input_file_name())
)

stops_df = (
    spark.read.options(**read_options)
    .csv(f"{target_paths['stops']}/stops.txt")
    .withColumn("ingestion_date", lit(ingestion_date))
    .withColumn("ingested_at", current_timestamp())
    .withColumn("source_file", input_file_name())
)

stop_times_df = (
    spark.read.options(**read_options)
    .csv(f"{target_paths['stop_times']}/stop_times.txt")
    .withColumn("ingestion_date", lit(ingestion_date))
    .withColumn("ingested_at", current_timestamp())
    .withColumn("source_file", input_file_name())
)

calendar_df = (
    spark.read.options(**read_options)
    .csv(f"{target_paths['calendar']}/calendar.txt")
    .withColumn("ingestion_date", lit(ingestion_date))
    .withColumn("ingested_at", current_timestamp())
    .withColumn("source_file", input_file_name())
)

shapes_df = (
    spark.read.options(**read_options)
    .csv(f"{target_paths['shapes']}/shapes.txt")
    .withColumn("ingestion_date", lit(ingestion_date))
    .withColumn("ingested_at", current_timestamp())
    .withColumn("source_file", input_file_name())
)

print("Leitura dos arquivos concluída.")

# COMMAND ----------
# Logs rápidos de volumetria
print("Contagem de registros:")
print(f"routes: {routes_df.count()}")
print(f"trips: {trips_df.count()}")
print(f"stops: {stops_df.count()}")
print(f"stop_times: {stop_times_df.count()}")
print(f"calendar: {calendar_df.count()}")
print(f"shapes: {shapes_df.count()}")

# COMMAND ----------
# Escrita em Delta na camada Bronze
print("Gravando datasets em Delta Lake...")

delta_paths = {
    "routes": f"abfss://{container}@{storage_account}.dfs.core.windows.net/gtfs_routes",
    "trips": f"abfss://{container}@{storage_account}.dfs.core.windows.net/gtfs_trips",
    "stops": f"abfss://{container}@{storage_account}.dfs.core.windows.net/gtfs_stops",
    "stop_times": f"abfss://{container}@{storage_account}.dfs.core.windows.net/gtfs_stop_times",
    "calendar": f"abfss://{container}@{storage_account}.dfs.core.windows.net/gtfs_calendar",
    "shapes": f"abfss://{container}@{storage_account}.dfs.core.windows.net/gtfs_shapes",
}

(
    routes_df.write.format("delta")
    .mode("overwrite")
    .save(delta_paths["routes"])
)

(
    trips_df.write.format("delta")
    .mode("overwrite")
    .save(delta_paths["trips"])
)

(
    stops_df.write.format("delta")
    .mode("overwrite")
    .save(delta_paths["stops"])
)

(
    stop_times_df.write.format("delta")
    .mode("overwrite")
    .save(delta_paths["stop_times"])
)

(
    calendar_df.write.format("delta")
    .mode("overwrite")
    .save(delta_paths["calendar"])
)

(
    shapes_df.write.format("delta")
    .mode("overwrite")
    .save(delta_paths["shapes"])
)

print("Gravação em Delta concluída.")

# COMMAND ----------
# Validação final simples
print("Validando paths Delta gravados...")

for name, path in delta_paths.items():
    exists = path_exists(path)
    print(f"{name}: {'OK' if exists else 'NÃO ENCONTRADO'} -> {path}")

print("=== GTFS STATIC INGESTION END ===")

# COMMAND ----------
# Limpeza local
print("Limpando arquivos temporários locais...")
cleanup_local_paths(zip_local_path, extract_base_local)
print("Limpeza concluída.")